<a href="https://colab.research.google.com/github/roxanapodean/Disertatie/blob/master/NN_binaryClass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow pandas scikit-learn

In [ ]:
import numpy as np
import tensorflow as tf

from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler

In [ ]:
np.random.seed(0)
tf.random.set_seed(0)

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving mult_10%EEC1_10%TCO1_Inputs.asc to mult_10%EEC1_10%TCO1_Inputs.asc


In [ ]:
#filename = "mult_100%EEC1_100%TCO1_Inputs.asc"
#filename = "mult_50%EEC1_100%TCO1_Inputs.asc"
#filename = "mult_50%EEC1_50%TCO1_Inputs.asc"
#filename = "mult_25%EEC1_100%TCO1_Inputs.asc"
#filename = "mult_25%EEC1_25%TCO1_Inputs.asc"
#filename = "mult_10%EEC1_100%TCO1_Inputs.asc"
filename = "mult_10%EEC1_10%TCO1_Inputs.asc"

with open(filename, "r") as f:
    lines = f.readlines()

lines = [line.strip() for line in lines if line.strip() != ""]

rows = []
outputs = []
max_inputs = 0

for line in lines:
    values = [float(x.strip()) for x in line.split(",")]

    output = values[-1]

    # converteste atacul 2 -> 1
    if output == 2:
        output = 1

    outputs.append(output)

    input_row = values[:-1]
    rows.append(input_row)

    max_inputs = max(max_inputs, len(input_row))

inputs = np.zeros((len(rows), max_inputs))

for i, row in enumerate(rows):
    inputs[i, :len(row)] = row

outputs = np.array(outputs).astype(int)

scaler = StandardScaler()

inputs = scaler.fit_transform(inputs)

print(inputs.shape)
print(outputs.shape)
print(np.unique(outputs, return_counts=True))

(89605, 6)
(89605,)
(array([0, 1]), array([81453,  8152]))


In [ ]:
n = len(inputs)

train_end = int(0.30 * n)
val_end = int(0.40 * n)

x_train = inputs[:train_end]
y_train = outputs[:train_end]

x_val = inputs[train_end:val_end]
y_val = outputs[train_end:val_end]

x_test = inputs[val_end:]
y_test = outputs[val_end:]

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=6,
    batch_size=256,
    verbose=1
)

Epoch 1/6
106/106 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9049 - loss: 0.3775 - val_accuracy: 0.9227 - val_loss: 0.2058
Epoch 2/6
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9616 - loss: 0.1315 - val_accuracy: 0.9699 - val_loss: 0.0853
Epoch 3/6
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9811 - loss: 0.0665 - val_accuracy: 0.9802 - val_loss: 0.0554
Epoch 4/6
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9872 - loss: 0.0447 - val_accuracy: 0.9861 - val_loss: 0.0378
Epoch 5/6
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9899 - loss: 0.0319 - val_accuracy: 0.9901 - val_loss: 0.0281
Epoch 6/6
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9925 - loss: 0.0247 - val_accuracy: 0.9912 - val_loss: 0.0225


In [ ]:
y_prob = model.predict(x_test)
y_pred = (y_prob > 0.5).astype(int).flatten()

1681/1681 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [ ]:
C = confusion_matrix(y_test, y_pred, labels=[0, 1])

print("Confusion matrix:")
print(C)

TN = C[0, 0]
FP = C[0, 1]
FN = C[1, 0]
TP = C[1, 1]

TNR = TN / (TN + FP) * 100 if (TN + FP) != 0 else 0
TPR = TP / (TP + FN) * 100 if (TP + FN) != 0 else 0
FNR = FN / (TP + FN) * 100 if (TP + FN) != 0 else 0
FPR = FP / (TN + FP) * 100 if (TN + FP) != 0 else 0

accuracy = (TP + TN) / (TP + TN + FP + FN) * 100
precision = TP / (TP + FP) * 100 if (TP + FP) != 0 else 0
f1_score = (2 * TP) / (2 * TP + FP + FN) * 100 if (2 * TP + FP + FN) != 0 else 0

print("\nMetric,Value")
print(f"TP:{TP}")
print(f"TN:{TN}")
print(f"FP:{FP}")
print(f"FN:{FN}")
print(f"TPR:{TPR:.2f}")
print(f"TNR:{TNR:.2f}")
print(f"FPR:{FPR:.2f}")
print(f"FNR:{FNR:.2f}")
print(f"Accuracy:{accuracy:.2f}")
print(f"Precision:{precision:.2f}")
print(f"F1Score:{f1_score:.2f}")

Confusion matrix:
[[48868     0]
 [  518  4377]]

Metric,Value
TP:4377
TN:48868
FP:0
FN:518
TPR:89.42
TNR:100.00
FPR:0.00
FNR:10.58
Accuracy:99.04
Precision:100.00
F1Score:94.41
